# 01 – Data Cleaning

This notebook covers loading the CKD dataset, inspecting data quality, handling missing values, and removing duplicates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────
TRAIN_PATH = '../dataset/Training_CKD_dataset.csv'
TEST_PATH  = '../dataset/Testing_CKD_dataset.csv'

# ── Load data ──────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print('Training set shape :', train_df.shape)
print('Testing  set shape :', test_df.shape)

## 1. First Look at the Data

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.describe().T.style.format('{:.3f}').background_gradient(cmap='Blues', axis=0)

## 2. Missing Values Analysis

In [ ]:
# ── Missing value counts ───────────────────────────────────────────────────
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count':      missing,
    'Missing Percentage': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

if missing_report.empty:
    print('✅ No missing values found in the training set!')
else:
    display(missing_report)

In [ ]:
# ── Visualise missing values ───────────────────────────────────────────────
total_missing = train_df.isnull().sum().sum()
print(f'Total missing cells : {total_missing}')
print(f'Total cells         : {train_df.size}')
print(f'Missing percentage  : {total_missing / train_df.size * 100:.4f}%')

## 3. Duplicate Row Detection

In [ ]:
dup_count = train_df.duplicated().sum()
print(f'Duplicate rows in training set: {dup_count}')

if dup_count > 0:
    train_df = train_df.drop_duplicates()
    print(f'Cleaned training set shape: {train_df.shape}')

## 4. Categorical Column Detection

In [ ]:
cat_cols = train_df.select_dtypes(include='object').columns.tolist()
num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()

print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
print(f'Numerical  columns  ({len(num_cols)}): {num_cols}')

In [ ]:
# ── Unique values for each categorical column ─────────────────────────────
for col in cat_cols:
    print(f'{col:35s}: {train_df[col].unique()}')

## 5. Categorical Encoding

In [ ]:
# ── Encode binary Yes/No columns ──────────────────────────────────────────
BINARY_COLS = ['Diabetes', 'Hypertension', 'Smoking_Status', 'Family_History_Kidney']

train_enc = train_df.copy()
for col in BINARY_COLS:
    if col in train_enc.columns:
        train_enc[col] = train_enc[col].map({'Yes': 1, 'No': 0})

# ── Encode the Target column ──────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_enc['Target_enc'] = le.fit_transform(train_enc['Target'])

print('Class mapping:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

## 6. Data Type Summary After Cleaning

In [ ]:
print('Data types after encoding:')
print(train_enc.dtypes)
print(f'\nFinal cleaned training shape: {train_enc.shape}')
print('\n✅ Data cleaning complete!')